# Contrastive Learning

[[paper]](https://arxiv.org/pdf/2002.05709)

**Contrastive Learning** — парадигма обучения, целью которой построение <u>представлений</u> (embeddings), при которой модель учат различать похожие и непохожие примеры. В отличие от обучения с учителем, мы здесь уходим от "абсолютного" описания объектов к "сравнительному" (похож / не похож). 

Человеческая разметка может применяться, но в большинстве случаев она как раз не нужна и её можно заменить автоматизированной. Contrastive Learning - это частный случай Self-Supervised Learning (обучения по "синтетически" собраным примерам), где акцент на контрастность (умение отличать).

__История__<br>
- в 2006 Ян Лекун описал принцип в рамках решения задачи распознавания лиц
- в 2015 в рамках разработки модели FaceNet появилась функция Triplet loss
- в ислледовательской работе CPC обосновали сипольщование контрастивной метрики InfoNCE
- в рамках работы SimCLR (2020) предложили метрику (назвали зачем-то NT-XENT)

Видно, что буст популярности был в основном в домене Computer Vision, там была самая явная потребность в нем. Но принцип перекочевал в другие домены, и применим к обучению любых моделей, требующих явной генерации представлений (все, где есть pre-training)

__Мотивация__<br>
Supervised Learning для получения внутренних описаний (эмбедингов) - это хорошо, но 
1) дорого
2) возможно только в парадигме end-to-end обучения, где есть конкретная внешняя задача => по определению не универсально

Хочется иметь дешевый и универсальный способ. Раньше таким занимались автоэнкодеры, но они слишком на никоуровенвый шум, устарели, их как раз заменили обучаемые модели.

__Идея:__ давайте учить так, чтобы представления "положительных" пар (похожих примеров) были максимально близки в пространстве эмбеддингов, а представления "отрицательных" пар (непохожих примеров) были максимально удалены друг от друга. Это заставляет модель улавливать существенные семантические признаки данных, игнорируя при этом незначительные вариации

<img src="img/contrastive_learning/contrastive1.png" width=500>

__Обсуждение__<br>
Два принципа контрастивного обучения:<br>
- вместо обучения модели относить точку к классу {класс 1, класс 2 ... класс K} идет обучение модели определять "свой" / "чужой" (причем "свой" тут гибкое понятие)<br>эффективнее, так как вместо абсолютных фичей учатся относительные как соотноятся два объекта чтобы приналлежат одному классу
- неразрывно связано с self-supervised подходов<br>эффективно на масштабе

Self-supervised Learning - это гибридный подход к обучению. Он unsupervised, поскольку мы руками ничего не размечаем и рабтает с сырыми данными, и он же supervised, поскольку тем не менее можно дешево выделить из этих данных какой-то сигнал (пара объектов точно "своя", если аугментирована из одного образа)

Contrastive Learning не просто а новый стандарт pre-training обучения

Эмбединги могут обучаться в рамках какой-то задачи, а могут сами по себе. Результаты (в частности SimCLR) показывают, что в отличие от эмбедингов, обученных c учителем под end-to-end задачу, грамотно обученные self-supervised эмбединги получаются <u>более робастными и универсальными</u> и показывают лучший результаты для большего спектра end-to-end задач

Для SimSLR это немного контринтуитивно, поскольку ground-truth сигнал, который должен направлять обучние, тут явно слабее - мы же ориентируемся на сильно синтетическую разметку - повороты одной и той же картнки. Две разные картинки из одного класса были бы лучше. Тем не менее даже это хорошо работает.

<img src="img/contrastive_learning/contrastive3.png" width=500>

Но в приложениях сигнал бывает и более естественным
- Vision: две картинки сгенерированы из одного изображения
- Text: два предложения сгенерированы их одного промпта
- Facial Recognition: у одного человека несколько фотографий

Он может быть даже мультимодальным (объекты разных типов)
- Audio: текст принадлежит изображению
- Multimodel: текст принадлежит картинке
- Recommendations: товар были куплен пользователем


__Иллюстрация__<br>
Есть 100 классов. Обучить хорошее представление = сделать так, чтобы представления классов кластеризовались в однородные облака

<img src="img/contrastive_learning/contrastive2.png" width=500>

__Loss functions__<br>
Есть несоклько функций потерь, используемые в Contrastive Learning. Принцип везде одинковый - меряем кросс-энтропию между предсказанием и ground truth на паре свой-чужой

Примеры функций<br>
- __Triplet Loss__<br>идея: для выбранной anchor точки расстояние до своего примера $f(p_i)$ должно быть как можно меньше, чем до чужого  $f(n_i)$ + порог $\alpha$, итого  ошибка<br>$$L_{Triplet} = \sum_{i=1}^{N} \max(0, \|f(a_i) - f(p_i)\|^2 - \|f(a_i) - f(n_i)\|^2 + \alpha) \rightarrow \min$$<br>минус этой метрики - чужие примеры часто слишком просто отделять<br><br>
- __InfoNCE__<br>идея: давайте брать много случайных негативных примеров, предложена в рамках Contrastive Predictive Coding (CPC), имеет теоретическую основу, скалярное произведение<br>$$L_{InfoNCE} = -\mathbb{E} \left[ \log \frac{\exp(f(x, c))}{\sum_{j=1}^{K} \exp(f(x_j, c))} \right]$$<br><br>
- __NT-Xent__<br>идея: в качестве негатива берем все примеры из батча<br>название идиотское, по факту это просто CE, предложена в рамках модели SimCLR, обычно использует косинусное расстояние<br>$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(\boldsymbol{z}_i, \boldsymbol{z}_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(\boldsymbol{z}_i, \boldsymbol{z}_k) / \tau)}$$<br>названо идиотским названием Normalized Temperature-scaled Cross-Entropy

__Что еще:__<br>
- SupCon (supervised Contrastive Learning) = если есть разметка, то в качестве положительных пар берем ещё все инстансы класса<br> 
- BYOL, W-MSE = вариации обучения вообще без негативных примеров:

Связанные термины<br>
- Metric Learning: задача поиска такого кодирования, которое сохраняло бы какое-то полезное свойство объекта относительно других

__Общий обзор__<br>
*   Supervised Learning: размеченные данные, конкретная задача, высокие результаты, большие объемы ручной разметки.
*   Unsupervised Learning: моделируют структуру, но редко обучали эмбеддинги, хорошо переносящиеся на сложные Downstream-задачи.
*   Self-Supervised Learning:
    *  Автоэнекодеры (1980-е) учились восстанавливать входные данные из сжатого представления
    *  Генеративные модели (VAE (2013), GAN (2014)): Учились генерировать новые данные. Хотя их энкодеры могли создавать представления, эти представления не всегда были оптимальны для дискриминационных задач.
    *  Специфические Pretext-задачи: Например, в компьютерном зрении предсказание поворота изображения, решение головоломок (Jigsaw Puzzles (2016)), заполнение закрашенных областей (Context Encoders (2016)), или в NLP — предсказание слова по контексту (Word2Vec (2013)) и маскирование слов (BERT (2018)). Эти методы были эффективны, но требовали разработки уникальной Pretext-задачи для каждого домена, и их представления часто уступали по качеству Contrastive Learning.

Под Contrastive Learning понимают просто универсальную схему self-supervised обучения, которая подходит под любые задачи

 

__Архитектура__<br>
Типичная архитектура модели обучаемой по принципу Contrastive Learning включает:
1.  Encoder `f` отображает входные данные `x` в низкоразмерное скрытое представление `h = f(x)`.
2.  Projection Head `g`: Нелинейная преобразует `h` в пространство `z = g(h)`<br>зачем - `z` может быть более "гибким" для Contrastive Loss, в то время как `h` сохраняет более общую информацию для Downstream-задач.
3.  Contrastive Loss сравнивает  представления
4.  Data Augmentation: генератор "положительных" пар из одного и того же исходного примера `x`

__Алгоритм обучения__<br>
На примере SimCLR (2020):
1.  из случайного объекта выборки `x` генерируем два новых объекта `x_i = t_i(x)` и `x_j = t_j(x)`<br>Эта пара (`x_i`, `x_j`) - positive пример
2.  повторяем так для N точек
3.  "не свои" сгенерированные точки - это negative примеры
4.  считаем эмбединги `h_i = f(x_i)` и `h_j = f(x_j)` и проецируем в эмбеддинги поменьше `z_i = g(h_i)` и `z_j = g(h_j)`.
5.  вычисление сходства: Вычисляется мера сходства между всеми парами эмбеддингов в батче. Обычно используется косинусное сходство или скалярное произведение (dot product).
6.  Contrastive Loss: для каждой пары считаем NT-Xent:
        $$ L_{i,j} = - \log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{k \neq i} \exp(\text{sim}(z_i, z_k) / \tau)} $$
        где $\text{sim}(u, v)$ — функция сходства (например, косинусное), $\tau$ — температурный параметр, регулирующий чувствительность к отрицательным примерам, а $\mathbb{1}_{k \neq i}$ указывает на то, что $k \neq i$.
7.  минимизируеум средний Contrastive Loss по всему батчу с использованием стандартных оптимизаторов (например, Adam, SGD)

__Алгоритм инференса__<br>
После завершения обучения:
1.  отбрасывается Projection head за ненадобностью
2.  генерируем эмбеддинги для новых данных
3.  решаем downstream-задачу

__Результаты__<br>
- self-supervised pre-training по принципу contrastive learning работает немногим хуже supervised обучения
- если к масштабному pre-training добавить всего <u>1%</u> supervised fine-tuning, он ставится даже <u>лучше</u> полного supervised training
- SimCLR (2020) на ImageNet дал прирост +7%

 __Критика:__<br>
- Contrastive Learning pre-training дешевый в терминах разметки, в терминах CPU он дороже - там квадратичные вычисления (нужно посчитать каждый с каждым) + требует больших батчей (до 8K для картинок)
- модель не видит соседние с анкором примеры, только аугментированные с ним самим<br>
Есть альтернативы: swAV, DINO

Некоторые эмбединги, построенные по принципу Contrastive Learning:
- SimCSE<br>аугментация = Dropout весов внутри энкодера
- E5<br>pre-training на дешево предразмеченых парах текстов из интернета + fine-tuning на датасете релевантности
- OPenaI Text Embeddings<br>соседние куски текста (?)
- CLIP<br>картинка и подпись к ней один класс
- BGE эмбединги для RAG<br>

Универсальные эмбединги сравниваются на бенчмарке __MTEB__ Massive Text Embedding Benchmark. Включает 50 рамеченых датасетов


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F

# Define the Encoder network (e.g., a simple CNN for illustrative purposes)
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.fc = nn.Linear(128 * 8 * 8, 256)  # Assuming input images are 32x32

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Define the Projection Head
class ProjectionHead(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ProjectionHead, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Contrastive Loss (NT-Xent Loss)
def contrastive_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.size(0)
    z = torch.cat([z_i, z_j], dim=0)
    sim_matrix = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)
    sim_matrix = sim_matrix / temperature

    # Create labels for positive pairs
    labels = torch.cat([torch.arange(batch_size) for _ in range(2)], dim=0)
    labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()

    # Mask to remove self-similarity
    mask = torch.eye(labels.size(0), dtype=torch.bool).to(z.device)
    labels = labels[~mask].view(labels.size(0), -1)
    sim_matrix = sim_matrix[~mask].view(sim_matrix.size(0), -1)

    # Compute the loss
    loss = F.cross_entropy(sim_matrix, labels)
    return loss

# Data augmentation
transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

# Load CIFAR-10 dataset
dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=2)

# Initialize the networks
encoder = SimpleCNN()
projection_head = ProjectionHead(input_dim=256, output_dim=128)

# Optimizer
optimizer = optim.Adam(list(encoder.parameters()) + list(projection_head.parameters()), lr=0.001)

# Training loop
for epoch in range(10):  # Train for 10 epochs
    for (x, _) in dataloader:
        # Generate two augmented views of each image
        x_i = transform(x)
        x_j = transform(x)

        # Encode and project
        h_i = encoder(x_i)
        h_j = encoder(x_j)
        z_i = projection_head(h_i)
        z_j = projection_head(h_j)

        # Compute contrastive loss
        loss = contrastive_loss(z_i, z_j)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/10], Loss: {loss.item():.4f}')

# After training, discard the projection head
# Use the encoder for downstream tasks
``` 

### Key Points:
- **Encoder**: A simple CNN is used to map input images to a latent space.
- **Projection Head**: A small MLP that projects the latent representation into a space where contrastive loss is computed.
- **Contrastive Loss**: NT-Xent loss is used to bring positive pairs closer and push negative pairs apart.
- **Data Augmentation**: Random transformations are applied to create positive pairs from the same image.
- **Training Loop**: The model is trained using the contrastive loss, optimizing both the encoder and projection head.

This example illustrates the core concepts of contrastive learning, focusing on the creation of positive and negative pairs, and the use of a projection head to facilitate the learning of discriminative embeddings.